In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
from peft import PeftConfig, PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from utils import (
    build_binary_relevance_maps,
    build_eval_items,
    build_global_corpus,
    build_sorted_pools_global,
    load_triplets,
    report,
)

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = os.path.join(PROJECT_ROOT, "dataset", "hard_triplets_tfidf_wikidata.json")
QWEN_LORA_PATH = os.path.join(PROJECT_ROOT, "../checkpoints", "qwen_2.5_wikidata_finetuned")

TOPKS = [3, 5, 10, 20]
QWEN_FT_ENCODE_BATCH = 8
QWEN_FT_MAX_LENGTH = 768
DEVICE_FT = "cuda" if torch.cuda.is_available() else "cpu"

raw_data = load_triplets(DATA_PATH)
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Loaded {len(raw_data)} items")
print(f"Global corpus size: {len(global_corpus)}")
print(f"Eval anchors with non-empty positives: {len(eval_items)}")
print(f"LoRA path: {QWEN_LORA_PATH}")

Loaded 186 items
Global corpus size: 2229
Eval anchors with non-empty positives: 186
LoRA path: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/evaluation/../checkpoints/qwen_2.5_wikidata_finetuned


In [3]:
def format_sparql_to_text_prompt(sparql: str) -> str:
    return (
        "You are an expert in knowledge graphs.\n\n"
        "### Instruction\n"
        "Convert the SPARQL query into a clear natural language question.\n\n"
        f"### SPARQL Query\n{sparql}\n\n"
        "### Natural Text\n"
    )


def l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp(min=eps)


class QwenSPARQLEncoder:
    def __init__(self, lora_path: str, device: str = DEVICE_FT):
        self.device = device
        peft_cfg = PeftConfig.from_pretrained(lora_path)
        self.base_model_name = peft_cfg.base_model_name_or_path

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(lora_path)
        except Exception:
            self.tokenizer = AutoTokenizer.from_pretrained(self.base_model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        if device == "cuda":
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            try:
                base_model = AutoModelForCausalLM.from_pretrained(
                    self.base_model_name,
                    quantization_config=bnb_config,
                    device_map={"": torch.cuda.current_device()},
                )
            except torch.cuda.OutOfMemoryError:
                base_model = AutoModelForCausalLM.from_pretrained(
                    self.base_model_name,
                    quantization_config=bnb_config,
                    device_map="auto",
                    llm_int8_enable_fp32_cpu_offload=True,
                )
        else:
            base_model = AutoModelForCausalLM.from_pretrained(
                self.base_model_name,
                torch_dtype=torch.float32,
            ).to(device)

        self.model = PeftModel.from_pretrained(base_model, lora_path, is_trainable=False)
        self.model.eval()

    def build_encoder_prompt(self, sparql: str) -> str:
        return format_sparql_to_text_prompt(sparql)

    @torch.no_grad()
    def encode_sparql(self, sparql_list, batch_size: int = QWEN_FT_ENCODE_BATCH, max_length: int = QWEN_FT_MAX_LENGTH):
        all_embeddings = []
        model_device = next(self.model.parameters()).device

        for start in tqdm(range(0, len(sparql_list), batch_size), desc="Qwen2.5 LoRA encode"):
            batch = sparql_list[start : start + batch_size]
            prompts = [self.build_encoder_prompt(s) for s in batch]
            toks = self.tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            )
            toks = {k: v.to(model_device) for k, v in toks.items()}

            outputs = self.model(
                **toks,
                output_hidden_states=True,
                return_dict=True,
            )
            last_hidden = outputs.hidden_states[-1]
            attn = toks["attention_mask"].unsqueeze(-1).float()
            pooled = (last_hidden * attn).sum(dim=1) / attn.sum(dim=1).clamp(min=1e-9)
            pooled = l2_normalize(pooled).to(torch.float32)
            all_embeddings.append(pooled.cpu())

        return torch.cat(all_embeddings, dim=0).numpy()

In [4]:
if not os.path.isdir(QWEN_LORA_PATH):
    print(f"Skip fine-tuned Qwen: not found at {QWEN_LORA_PATH}")
else:
    print(f"Loading fine-tuned Qwen2.5 LoRA from {QWEN_LORA_PATH} ...")
    qwen_ft_encoder = QwenSPARQLEncoder(QWEN_LORA_PATH, device=DEVICE_FT)
    qwen_ft_mat = qwen_ft_encoder.encode_sparql(
        global_corpus,
        batch_size=QWEN_FT_ENCODE_BATCH,
        max_length=QWEN_FT_MAX_LENGTH,
    )
    qwen_ft_map = {q: qwen_ft_mat[i] for i, q in enumerate(global_corpus)}
    qwen_ft_sorted = build_sorted_pools_global(qwen_ft_map, eval_items, global_corpus)
    report(
        "Fine-tuned Qwen2.5 LoRA (SPARQL->text, wikidata)",
        qwen_ft_sorted,
        eval_items,
        relevance_scores_per_item,
        TOPKS,
    )

Loading fine-tuned Qwen2.5 LoRA from /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/evaluation/../checkpoints/qwen_2.5_wikidata_finetuned ...


Loading weights:   1%|          | 2/339 [00:00<00:52,  6.37it/s]/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Qwen2.5 LoRA encode: 100%|██████████| 279/279 [03:07<00:00,  1.48it/s]



Fine-tuned Qwen2.5 LoRA (SPARQL->text, wikidata)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0914  0.0185  0.0291  0.0959  0.1129    0.0914  0.0895
@5     0.0935  0.0317  0.0448  0.1037  0.1452    0.0935  0.0921
@10    0.0796  0.0569  0.0619  0.1117  0.2043    0.0796  0.0877
@20    0.0457  0.0662  0.0509  0.1133  0.2312    0.0457  0.0708
